In [ ]:
################################################################################
# BiTimelyGPT - Factorial Experiment (Simulated Data)
################################################################################

#####################################################################
# Imports and Setup
#####################################################################
!pip install rpy2==3.5.1 prettytable

import itertools
import time
import torch
import numpy as np
import pandas as pd
import os
import math

from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc

%load_ext rpy2.ipython
from rpy2.robjects import pandas2ri
pandas2ri.activate()

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/BiTimelyGPT-main/BiTimelyGPT')

from data.data_pipeline import custom_collate
from models.BiTimelyGPT import BiTimelyGPT
from layers.optimization import get_linear_schedule_with_warmup, AdamW
from layers.heads import PretrainHead, ClfHead

set_seed = 123
torch.manual_seed(set_seed)
np.random.seed(set_seed)

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

################################################################################
# Define the R DGP script path and load into Colab
################################################################################

%%R
source("drive/MyDrive/sim_dgp.R")


In [ ]:
################################################################################
# specialized pipeline for Sim Data
################################################################################

class SimulatedDataset(torch.utils.data.Dataset):
    """
    Each item in this dataset is (sequence_tensor, label_tensor),
    where sequence_tensor has shape [T_i, p], label_tensor is 0/1 (LongTensor).
    """
    def __init__(self, samples, labels):
        super().__init__()
        self.samples = samples
        self.labels  = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq = self.samples[idx]
        lbl = self.labels[idx]
        return seq, lbl


def load_simulated_data(df_vars, df_outcomes, config):
    """
    Builds a list of variable-length sequences + labels from:
      df_vars     => [ID, time, variable, value]
      df_outcomes => [ID, time, Y, true_prob]
    config.feature_list => e.g., ["X1","X2","X3"]
    Returns: samples (list of Tensors) and labels (LongTensor).
    """
    feature_map = {feat: i for i, feat in enumerate(config.feature_list)}

    samples = []
    labels = []

    subject_ids = df_vars["ID"].unique()
    for sid in subject_ids:
        sub_df  = df_vars[df_vars["ID"] == sid].copy()
        sub_out = df_outcomes[df_outcomes["ID"] == sid]

        if len(sub_out) == 1:
            label_val = sub_out["Y"].values[0]
        else:
            label_val = 0

        sub_df = sub_df.sort_values("time")

        seq_tensors = []
        for t_val, group in sub_df.groupby("time"):
            feats = torch.zeros(len(config.feature_list), dtype=torch.float)
            for _, row in group.iterrows():
                var_name = row["variable"]
                if var_name in feature_map:
                    idx = feature_map[var_name]
                    feats[idx] = row["value"]
            seq_tensors.append(feats)

        if len(seq_tensors) == 0:
            seq_tensor = torch.zeros(1, len(config.feature_list))
        else:
            seq_tensor = torch.stack(seq_tensors, dim=0)

        samples.append(seq_tensor)
        labels.append(label_val)

    labels = torch.tensor(labels, dtype=torch.long)
    return samples, labels


def simulated_data_pipeline(df_vars, df_outcomes, config, train_ratio=0.8, shuffle=True):
    """
    Splits into train/test sets using 'train_ratio'. Returns:
      train_dataset, train_loader, test_dataset, test_loader
    """
    samples, labels = load_simulated_data(df_vars, df_outcomes, config)
    full_dataset    = SimulatedDataset(samples, labels)

    N = len(full_dataset)
    train_size = int(train_ratio * N)
    test_size  = N - train_size

    if shuffle:
        generator = torch.Generator().manual_seed(123)
        train_dataset, test_dataset = torch.utils.data.random_split(
            full_dataset, [train_size, test_size], generator=generator
        )
    else:
        train_dataset, test_dataset = torch.utils.data.random_split(
            full_dataset, [train_size, test_size]
        )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=custom_collate
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=custom_collate
    )
    return train_dataset, train_loader, test_dataset, test_loader

In [ ]:
################################################################################
# Define a Configuration Class for the Model
################################################################################

class SimDataConfig:
    def __init__(self, p):
        self.feature_list = [f"X{i+1}" for i in range(p)]
        self.n_output = len(self.feature_list)
        self.n_clf_output = 2

        self.num_layers = 6
        self.num_heads = 2
        self.d_model = 64
        self.qk_dim = 64
        self.v_dim = 128
        self.ffn_proj_size = 256
        self.d_ff = 128
        self.dropout = 0.1
        self.activation = 'gelu'

        self.use_bias_in_msr = False
        self.use_bias_in_mlp = True
        self.use_bias_in_msr_out = False
        self.use_default_gamma = False
        self.forward_impl = 'parallel'
        self.chunk_size = 12
        self.seq_len = 128

        self.pretrain_batch_size = 128
        self.pretrain_learning_rate = 1e-4
        self.pretrain_warmup_steps = 100

        self.finetune_batch_size = 32
        self.finetune_learning_rate = 2e-4
        self.finetune_epochs = 50
        self.finetune_warmup_steps = 100
        self.gradient_clip = 1.0

        self.use_gpu = torch.cuda.is_available()
        self.use_grad_ckp = False
        self.use_grad_accum = True
        self.accum_steps = 2
        self.use_amp = False
        self.use_multi_gpu = False
        self.devices = '0'

        self.output_retentions = False

In [ ]:
################################################################################
# Utility: Train & Evaluate function with Pretraining + Fine-tuning
################################################################################
import rpy2.robjects as ro
from torch.utils.data import DataLoader, random_split
import math
import numpy as np
import torch
import os
from torch import nn
from layers.optimization import get_linear_schedule_with_warmup, AdamW
from models.BiTimelyGPT import BiTimelyGPT
from layers.heads import ClfHead

def run_single_scenario(
    N, p,
    regularity,
    missing_prop,
    num_sync_groups,
    process_types,
    use_lags,
    block_missing=True,
    block_length=1.5,
    outcome_time=10,
    master_seed=123
):
    """
    Simulate data in R.
     Split into 70% train, 15% val, 15% test.
     Pretrain BiTimelyGPT using a self-supervised objective on the training set.
     Fine-tune the pretrained BiTimelyGPT on the classification task (training set),
       using validation-based early stopping (AUC).
     Evaluate the best fine-tuned model on the test set (Loss, AUC, Acc, Prec, Rec, F1 @ threshold 0.5).
    """
    print(f"\n--- Running Scenario: N={N}, p={p}, reg={regularity}, miss={missing_prop}, sync={num_sync_groups}, lag={use_lags} ---")

    ro.r.assign("N", N)
    ro.r.assign("p", p)
    ro.r.assign("regularity", regularity)
    ro.r.assign("missing_prop", missing_prop)
    ro.r.assign("num_sync_groups", num_sync_groups)
    if isinstance(process_types, list):
        ro.r.assign("process_types", ro.StrVector(process_types))
    else:
        ro.r.assign("process_types", ro.StrVector([process_types]*p))
    ro.r.assign("use_lags", use_lags)
    ro.r.assign("block_missing", block_missing)
    ro.r.assign("block_length", block_length)
    ro.r.assign("outcome_time", outcome_time)
    ro.r.assign("master_seed", master_seed)

    ro.r('''
        sim_dataset <- simulate_experiment_dataset(
            N               = N,
            p               = p,
            freq_min        = 0.5,
            freq_max        = 3.0,
            regularity      = regularity,
            num_sync_groups = num_sync_groups,
            process_types   = process_types,
            missing_prop    = missing_prop,
            block_missing   = block_missing,
            block_length    = block_length,
            outcome_time    = outcome_time,
            beta            = c(0, seq(-1, 1, length.out = 30)), # Assuming p=30 for beta length
            use_lags        = use_lags,
            lag_hours       = ifelse(use_lags, 2, 0),
            master_seed     = master_seed
        )
    ''')

    df_vars_r     = ro.r('sim_dataset$vars_long')
    df_outcomes_r = ro.r('sim_dataset$outcome_df')

    df_vars       = df_vars_r
    df_outcomes   = df_outcomes_r

    df_vars["ID"]     = df_vars["ID"].astype(int)
    df_vars["time"]   = df_vars["time"].astype(float)
    df_outcomes["ID"] = df_outcomes["ID"].astype(int)
    df_outcomes["time"]= df_outcomes["time"].astype(float)
    df_outcomes["Y"] = df_outcomes["Y"].astype(int)

    mean_true_prob = df_outcomes["true_prob"].mean()
    prop_positive  = df_outcomes["Y"].mean()
    print(f"Data generated: {len(df_outcomes)} subjects, {prop_positive*100:.1f}% positive cases.")


    config = SimDataConfig(p)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    samples, labels = load_simulated_data(df_vars, df_outcomes, config)
    full_dataset    = SimulatedDataset(samples, labels)

    N_total = len(full_dataset)
    if N_total < 10:
         print(f"Warning: Very small dataset (N={N_total}). Skipping scenario.")
         return { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                  "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }

    N_train = int(0.70 * N_total)
    N_val   = int(0.15 * N_total)
    N_test  = N_total - N_train - N_val

    if N_train <= 0 or N_val <= 0 or N_test <= 0:
        print(f"Warning: Insufficient data for standard 70/15/15 split (N={N_total}). Attempting adjusted split.")
        N_train = max(1, N_total - 2)
        N_val = 1 if N_total > 1 else 0
        N_test = 1 if N_total > 2 else 0
        if N_train <= 0 or (N_val + N_test == 0):
             print(f"Error: Cannot create a valid train/val/test split for N={N_total}. Skipping scenario.")
             return { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                      "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }

    print(f"Splitting data: Train={N_train}, Val={N_val}, Test={N_test}")
    generator = torch.Generator().manual_seed(master_seed)
    train_dataset, val_test_dataset = random_split(full_dataset, [N_train, N_val + N_test], generator=generator)

    if N_val + N_test > 0 :
        val_dataset, test_dataset = random_split(val_test_dataset, [N_val, N_test], generator=generator)
    else:
        val_dataset = torch.utils.data.Subset(val_test_dataset, [])
        test_dataset = torch.utils.data.Subset(val_test_dataset, [])


    train_loader = DataLoader(
        train_dataset, batch_size=config.pretrain_batch_size, shuffle=True,
        num_workers=2, pin_memory=True, collate_fn=custom_collate
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config.finetune_batch_size, shuffle=False,
        num_workers=2, pin_memory=True, collate_fn=custom_collate
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config.finetune_batch_size, shuffle=False,
        num_workers=2, pin_memory=True, collate_fn=custom_collate
    )

    if len(train_loader) == 0:
        print("Error: Training loader is empty. Cannot proceed. Skipping scenario.")
        return { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                 "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }


    model = BiTimelyGPT(configs=config, head_type='pretrain').to(device)
    model.head_type = 'pretrain'
    print(f"Initialized model with Pretrain head. Parameters: {sum(p.numel() for p in model.parameters())}")


    if N <= 500:
        config.pretrain_epochs = 15
    elif N <= 5000:
        config.pretrain_epochs = 15
    else:
        config.pretrain_epochs = 5
    print(f"--- Starting Pretraining Phase ({config.pretrain_epochs} epochs based on N={N}) ---")

    pretrain_optimizer = AdamW(
        model.parameters(), lr=config.pretrain_learning_rate, weight_decay=0.01
    )
    num_pretraining_steps = len(train_loader) * config.pretrain_epochs
    pretrain_scheduler = get_linear_schedule_with_warmup(
        pretrain_optimizer, config.pretrain_warmup_steps, num_pretraining_steps
    )

    for epoch in range(config.pretrain_epochs):
        model.train()
        pretrain_losses = []
        pretrain_optimizer.zero_grad()

        for i, (batch_x, _, attention_mask) in enumerate(train_loader):
            batch_x, attention_mask = batch_x.to(device), attention_mask.to(device)

            loss = model(batch_x, y=None, retention_mask=None,
                         forward_impl=config.forward_impl, chunk_size=config.chunk_size)

            if torch.isnan(loss):
                print(f"Warning: NaN loss at pretrain epoch {epoch+1}, batch {i}. Skipping update.")
                continue

            scaled_loss = loss / config.accum_steps
            scaled_loss.backward()

            if (i + 1) % config.accum_steps == 0 or (i + 1) == len(train_loader):
                pretrain_optimizer.step()
                pretrain_scheduler.step()
                pretrain_optimizer.zero_grad()

            pretrain_losses.append(loss.item())

        avg_pretrain_loss = np.mean(pretrain_losses) if pretrain_losses else np.nan
        print(f'Pretrain Epoch {epoch+1}/{config.pretrain_epochs} -- Avg Loss: {avg_pretrain_loss:.4f}')

    print("--- Pretraining Phase Complete ---")

    print(f"--- Switching to Fine-tuning Phase ({config.finetune_epochs} epochs) ---")
    model.head = ClfHead(config.d_model, config.n_clf_output).to(device)
    model.head_type = 'clf'
    print(f"Model head switched to ClfHead. Total params now: {sum(p.numel() for p in model.parameters())}")

    optimizer = AdamW(model.parameters(), lr=config.finetune_learning_rate, weight_decay=0.01)
    num_finetune_steps = len(train_loader) * config.finetune_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, config.finetune_warmup_steps, num_finetune_steps
    )
    criterion = nn.CrossEntropyLoss()

    best_val_auc = -1.0
    patience = 10
    epochs_no_improve = 0
    warmup_epochs = 2
    best_model_path = f"best_model_temp_{master_seed}.pt"

    print("--- Starting Fine-tuning Training Loop ---")
    for epoch in range(config.finetune_epochs):
        model.train()
        train_losses = []
        optimizer.zero_grad()

        for i, (batch_x, batch_y, attention_mask) in enumerate(train_loader):
            batch_x, batch_y, attention_mask = (
                batch_x.to(device), batch_y.to(device), attention_mask.to(device)
            )

            assert model.head_type == 'clf', "Model head_type should be 'clf'"
            hidden_states = model.conv_subsampling(batch_x)[0]
            hidden_states = model.input_projection(hidden_states)
            if attention_mask.shape[1] > hidden_states.shape[1]:
                subsampling_factor = batch_x.shape[1] // hidden_states.shape[1]
                attention_mask = attention_mask[:, ::subsampling_factor]
                attention_mask = attention_mask[:, :hidden_states.shape[1]]

            for block in model.blocks:
                out = block(hidden_states, retention_mask=attention_mask,
                            forward_impl=config.forward_impl, chunk_size=config.chunk_size)
                hidden_states = out[0]

            X = model.ln_f(hidden_states)
            logits = model.head(X)

            loss = criterion(logits, batch_y)
            scaled_loss = loss / config.accum_steps
            scaled_loss.backward()

            if (i + 1) % config.accum_steps == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            train_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses) if train_losses else np.nan

        if len(val_loader) == 0:
            print(f"[Fine-tune Epoch {epoch+1}/{config.finetune_epochs}] TrainLoss={avg_train_loss:.4f}, Skipping Validation (empty loader)")
            if (epoch + 1) % 10 == 0:
                 torch.save(model.state_dict(), best_model_path)
                 print(f"Saved model state at epoch {epoch+1} (no validation set).")
            continue

        model.eval()
        val_probs = []
        val_labels = []
        val_losses = []

        with torch.no_grad():
            for (batch_x, batch_y, attention_mask) in val_loader:
                batch_x, batch_y, attention_mask = (
                    batch_x.to(device), batch_y.to(device), attention_mask.to(device)
                )
                hidden_states = model.conv_subsampling(batch_x)[0]
                hidden_states = model.input_projection(hidden_states)
                if attention_mask.shape[1] > hidden_states.shape[1]:
                    subsampling_factor = batch_x.shape[1] // hidden_states.shape[1]
                    attention_mask = attention_mask[:, ::subsampling_factor]
                    attention_mask = attention_mask[:, :hidden_states.shape[1]]
                for block in model.blocks:
                    out = block(hidden_states, retention_mask=attention_mask,
                                forward_impl=config.forward_impl, chunk_size=config.chunk_size)
                    hidden_states = out[0]
                X_val = model.ln_f(hidden_states)
                logits = model.head(X_val)

                loss_val = criterion(logits, batch_y)
                val_losses.append(loss_val.item())
                probs = torch.softmax(logits, dim=1)[:, 1]
                val_probs.extend(probs.cpu().numpy())
                val_labels.extend(batch_y.cpu().numpy())

        avg_val_loss = np.mean(val_losses) if val_losses else np.nan
        val_auc = np.nan
        if len(np.unique(val_labels)) > 1:
            try:
                fpr_val, tpr_val, _ = roc_curve(val_labels, val_probs)
                val_auc = auc(fpr_val, tpr_val)
            except Exception as e:
                print(f"Warning: Could not calculate validation AUC: {e}")
        else:
             val_auc = 0.0
             print(f"Validation AUC calculation skipped: Only one class present ({np.unique(val_labels)}).")

        print(f"[Fine-tune Epoch {epoch+1}/{config.finetune_epochs}] "
              f"TrainLoss={avg_train_loss:.4f}, ValLoss={avg_val_loss:.4f}, ValAUC={val_auc:.4f}")

        if epoch + 1 > warmup_epochs and not np.isnan(val_auc):
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"==> Val AUC improved to {best_val_auc:.4f}. Fine-tuned model saved.")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"Early stopping fine-tuning (no val AUC improvement in {patience} epochs).")
                    break

    print("--- Evaluating best fine-tuned model on Test Set ---")
    if len(test_loader) == 0:
        print("Skipping Test Set evaluation (empty loader).")
        results_dict = { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                         "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }
        if os.path.exists(best_model_path):
            os.remove(best_model_path)
        return results_dict


    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print(f"Loaded best fine-tuned model from {best_model_path}")
    else:
        print("Warning: No best fine-tuned model saved (e.g., no validation improvement or no val set). Using the last model state for testing.")
        model.eval()

    all_probs = []
    all_labels = []
    test_losses = []

    with torch.no_grad():
        for (batch_x, batch_y, attention_mask) in test_loader:
            batch_x, batch_y, attention_mask = (
                batch_x.to(device), batch_y.to(device), attention_mask.to(device)
            )
            hidden_states = model.conv_subsampling(batch_x)[0]
            hidden_states = model.input_projection(hidden_states)
            if attention_mask.shape[1] > hidden_states.shape[1]:
                 subsampling_factor = batch_x.shape[1] // hidden_states.shape[1]
                 attention_mask = attention_mask[:, ::subsampling_factor]
                 attention_mask = attention_mask[:, :hidden_states.shape[1]]
            for block in model.blocks:
                out = block(hidden_states, retention_mask=attention_mask,
                            forward_impl=config.forward_impl, chunk_size=config.chunk_size)
                hidden_states = out[0]
            X_test = model.ln_f(hidden_states)
            logits = model.head(X_test)

            loss_test = criterion(logits, batch_y)
            test_losses.append(loss_test.item())
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_test_loss = float(np.mean(test_losses)) if test_losses else 0.0
    test_auc = 0.0
    test_acc = 0.0
    test_prec = 0.0
    test_rec = 0.0
    test_f1 = 0.0

    if all_labels and all_probs:
        try:
            fpr_test, tpr_test, _ = roc_curve(all_labels, all_probs)
            test_auc = auc(fpr_test, tpr_test)

            threshold = 0.5
            preds = (np.array(all_probs) >= threshold).astype(int)
            labels_arr = np.array(all_labels)

            tp = np.sum((preds == 1) & (labels_arr == 1))
            fp = np.sum((preds == 1) & (labels_arr == 0))
            fn = np.sum((preds == 0) & (labels_arr == 1))
            tn = np.sum((preds == 0) & (labels_arr == 0))

            test_prec = tp / (tp + fp + 1e-9)
            test_rec = tp / (tp + fn + 1e-9)
            test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec + 1e-9)
            test_acc = (tp + tn) / len(labels_arr) if len(labels_arr) > 0 else 0.0

        except ValueError as e:
            print(f"Could not calculate test metrics (AUC or others likely failed): {e}")
        except Exception as e:
             print(f"An unexpected error occurred during metric calculation: {e}")

    else:
        print("Skipping test metric calculation: No predictions or labels found.")

    print(f"Test Results: Loss={avg_test_loss:.4f}, AUC={test_auc:.4f}, F1={test_f1:.4f}")

    results_dict = {
        "test_loss":       avg_test_loss if not np.isnan(avg_test_loss) else 0.0,
        "auc":             test_auc,
        "accuracy":        test_acc,
        "precision":       test_prec,
        "recall":          test_rec,
        "f1":              test_f1,
        "mean_true_prob":  float(mean_true_prob) if not np.isnan(mean_true_prob) else 0.0,
        "prop_positive":   float(prop_positive) if not np.isnan(prop_positive) else 0.0
    }

    if os.path.exists(best_model_path):
        os.remove(best_model_path)

    return results_dict

In [ ]:
################################################################################
# 6) Main Loop: Full Factorial Over the 6 Factors
################################################################################

Ns = [200, 1000, 10000]
regularities = [0.2, 0.8]
missing_props = [0.1, 0.4]
sync_pattern_values = [1, "p"]
process_type_values = ["homogeneous", "mixed"]
outcome_dep_values  = ["direct", "lagged"]

scenario_grid = list(itertools.product(
    Ns,
    regularities,
    missing_props,
    sync_pattern_values,
    process_type_values,
    outcome_dep_values
))

all_results = []
overall_start = time.time()

for scenario in scenario_grid:
    (N,
     regularity,
     missing_prop,
     sync_pattern,
     process_type_flag,
     outcome_dep) = scenario

    p = 30
    if process_type_flag == "homogeneous":
        process_types = ["ar1"]*p
    else:
        repeats = math.ceil(p/3)
        big_array = (["ar1","rw","seasonal"] * repeats)[:p]
        process_types = big_array

    if sync_pattern == "p":
        num_sync_groups = max(1, int(round(p / 10)))
    else:
        num_sync_groups = 1

    if outcome_dep == "direct":
        use_lags = False
    else:
        use_lags = True

    print("===================================================================")
    print(f"Scenario => N={N}, reg={regularity}, missing={missing_prop}, "
          f"sync={sync_pattern}, process={process_type_flag}, outcome={outcome_dep}")
    print("===================================================================")

    start_time = time.time()
    results_dict = run_single_scenario(
        N=N,
        p=p,
        regularity=regularity,
        missing_prop=missing_prop,
        num_sync_groups=num_sync_groups,
        process_types=process_types,
        use_lags=use_lags,
        block_missing=True,
        block_length=1.5,
        outcome_time=10,
        master_seed=123
    )
    runtime = time.time() - start_time

    scenario_result = {
        "N":            N,
        "regularity":   regularity,
        "missing_prop": missing_prop,
        "sync_pattern": sync_pattern,
        "process_type": process_type_flag,
        "outcome_dep":  outcome_dep,
        "test_loss":    results_dict["test_loss"],
        "auc":          results_dict["auc"],
        "accuracy":     results_dict["accuracy"],
        "precision":    results_dict["precision"],
        "recall":       results_dict["recall"],
        "f1":           results_dict["f1"],

        "runtime_sec":  runtime,
        "mean_true_prob": results_dict["mean_true_prob"],
        "prop_positive":  results_dict["prop_positive"]
    }
    all_results.append(scenario_result)

overall_runtime = time.time() - overall_start
print(f"\nAll scenarios completed in {overall_runtime/60:.2f} minutes.\n")

################################################################################
# 7) Convert results to a DataFrame and save
################################################################################
df_results = pd.DataFrame(all_results)
df_results.sort_values(by=["N","regularity","missing_prop"], inplace=True)
print(df_results)

df_results.to_csv("drive/MyDrive/bitimelygpt_factorial_results_REP1.csv", index=False)
print("\nResults saved to bitimelygpt_factorial_results_REP1.csv")

Streaming output truncated to the last 5000 lines.
==> Val AUC improved to 0.6986. Fine-tuned model saved.
[Fine-tune Epoch 27/50] TrainLoss=0.6064, ValLoss=0.6094, ValAUC=0.7135
==> Val AUC improved to 0.7135. Fine-tuned model saved.
[Fine-tune Epoch 28/50] TrainLoss=0.5863, ValLoss=0.5996, ValAUC=0.7367
==> Val AUC improved to 0.7367. Fine-tuned model saved.
[Fine-tune Epoch 29/50] TrainLoss=0.5850, ValLoss=0.6001, ValAUC=0.7438
==> Val AUC improved to 0.7438. Fine-tuned model saved.
[Fine-tune Epoch 30/50] TrainLoss=0.5571, ValLoss=0.5551, ValAUC=0.7785
==> Val AUC improved to 0.7785. Fine-tuned model saved.
[Fine-tune Epoch 31/50] TrainLoss=0.5372, ValLoss=0.6514, ValAUC=0.7662
[Fine-tune Epoch 32/50] TrainLoss=0.5423, ValLoss=0.5563, ValAUC=0.7785
[Fine-tune Epoch 33/50] TrainLoss=0.5279, ValLoss=0.6611, ValAUC=0.7707
[Fine-tune Epoch 34/50] TrainLoss=0.5130, ValLoss=0.5393, ValAUC=0.8031
==> Val AUC improved to 0.8031. Fine-tuned model saved.
[Fine-tune Epoch 35/50] TrainLoss=0.4